In [ ]:
import qlib
from qlib.config import REG_CN

qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region=REG_CN
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from qlib.data.dataset import DatasetH
from qlib.contrib.data.handler import Alpha158
from qlib.contrib.model.gbdt import LGBModel

In [ ]:
dataset = DatasetH(
    handler={
        "class": "Alpha158",
        "module_path": "qlib.contrib.data.handler",
        "kwargs": {
            "start_time": "2015-01-01",
            "end_time": "2020-12-31",
            "fit_start_time": "2015-01-01",
            "fit_end_time": "2018-12-31",
            "instruments": "csi500",
        },
    },
    segments={
        "train": ("2015-01-01", "2018-12-31"),
        "valid": ("2019-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-12-31"),
    },
)

In [ ]:
model = LGBModel(
    loss="mse",
    learning_rate=0.05,
    num_leaves=210,
    n_estimators=200
)

model.fit(dataset)

In [ ]:
pred = model.predict(dataset)

pred.head()

In [ ]:
label_df = dataset.prepare("test", col_set="label")

label_df.head()

In [ ]:
df = pd.concat([pred, label_df], axis=1)

df.columns = ["pred", "label"]

df.head()

In [ ]:
ic_series = df.groupby(level="datetime").apply(
    lambda x: x["pred"].corr(x["label"])
)

ic_series.head()

In [ ]:
ic_mean = ic_series.mean()
ic_std = ic_series.std()

print("IC Mean:", ic_mean)
print("IC Std:", ic_std)

In [ ]:
plt.figure(figsize=(10,4))
ic_series.plot()
plt.title("Daily Information Coefficient")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
ic_series.hist(bins=50)
plt.title("IC Distribution")
plt.show()

In [ ]:
cols = list(df.columns)
print("Columns:", cols)

In [ ]:
ic_series = df.groupby(level="datetime").apply(
    lambda x: x["pred"].corr(x["label"], method="spearman")
)

In [ ]:
rolling_ic = ic_series.rolling(20).mean()

plt.figure(figsize=(12,5))
rolling_ic.plot()
plt.axhline(0, linestyle="--")
plt.title("Rolling 20-Day IC Mean")
plt.show()

In [ ]:
positive_ratio = (ic_series > 0).mean()
print("Positive IC ratio:", positive_ratio)

In [ ]:
df["quantile"] = df.groupby(level="datetime")["pred"].transform(
    lambda x: pd.qcut(x, 5, labels=False, duplicates="drop")
)

In [ ]:
quantile_returns = df.groupby(["datetime","quantile"])["label"].mean().unstack()

In [ ]:
import matplotlib.pyplot as plt

(quantile_returns.cumsum()).plot(figsize=(12,6))
plt.title("Cumulative Return by Quantile")
plt.show()

In [ ]:
long_short = quantile_returns[4] - quantile_returns[0]

long_short.cumsum().plot(figsize=(12,5))
plt.title("Long Top Quantile – Short Bottom Quantile")
plt.show()

In [ ]:
print("Long-Short Mean Return:", long_short.mean())
print("Long-Short Sharpe:", long_short.mean()/long_short.std())

In [ ]:
import pandas as pd

horizons = [1, 2, 3, 5, 10]
ic_decay = {}

for h in horizons:
    
    future_ret = df.groupby(level="instrument")["label"].shift(-h)
    
    temp = df.copy()
    temp["future_return"] = future_ret
    
    ic = temp.groupby(level="datetime").apply(
        lambda x: x["pred"].corr(x["future_return"], method="spearman")
    )
    
    ic_decay[h] = ic.mean()

In [ ]:
print(ic_decay)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(list(ic_decay.keys()), list(ic_decay.values()), marker='o')
plt.title("IC Decay Curve")
plt.xlabel("Prediction Horizon (days)")
plt.ylabel("IC")
plt.show()

Factor Turnover

In [ ]:
df["rank"] = df.groupby(level="datetime")["pred"].rank(pct=True)

In [ ]:
rank_shift = df.groupby(level="instrument")["rank"].shift(1)

turnover = (df["rank"] - rank_shift).abs()

In [ ]:
turnover_mean = turnover.mean()
print("Factor Turnover:", turnover_mean)